In [1]:
import polars as pl
import xarray as xr
import pandas as pd
import numpy as np
from datetime import date

In [7]:
# === пути (ПОМЕНЯЙ под себя) ===
STATIONS_PATH = "../data/weatherstation_data/stations_2000_2020_Russia.parquet"
# STATIONS_PATH = "../data/weatherstation_data/stations_2017_2020_Russia.parquet"
CMIP_PATH = "../data/cmip5_world_orig/sfcWindmax_day_cmip5_cmip5world_r1i1p1_1950-01-01-2020-11-13.nc"
OUT_PATH = "../data/weatherstation_data/cmip_station_2000_2020_Russia.parquet"
# OUT_PATH = "../data/weatherstation_data/cmip_station_2017_2020_Russia.parquet"


# === 1) читаем станции ===
stations = pl.read_parquet(STATIONS_PATH)

# sanity-check
assert set(stations.columns) >= {"STATION", "LATITUDE", "LONGITUDE"}
print("stations:", stations.height)
print(stations.head())

# === 2) открываем CMIP и режем по времени ===
ds = xr.open_dataset(CMIP_PATH)
da = ds["sfcWindmax"].sel(time=slice("2000-01-01", "2020-11-13"))
# da = ds["sfcWindmax"].sel(time=slice("2017-01-01", "2020-11-13"))


# приводим время к date (без времени)
times = pd.to_datetime(da["time"].values).date
n_time = len(times)
print("cmip time points:", n_time, "from", times[0], "to", times[-1])

# координаты сетки (монотонные 1D)
lat_grid = da["lat"].values
lon_grid = da["lon"].values

# === 3) функция поиска ближайшего индекса на монотонной сетке ===
def nearest_index(grid: np.ndarray, x: float) -> int:
    # grid должен быть отсортирован по возрастанию
    i = int(np.searchsorted(grid, x))
    if i <= 0:
        return 0
    if i >= len(grid):
        return len(grid) - 1
    # выбираем ближнюю из двух соседних
    return i if abs(grid[i] - x) < abs(grid[i-1] - x) else i-1

# === 4) извлечение рядов по станциям ===
st_pdf = stations.to_pandas()
rows = []

# загрузим da в память "лениво" не получится; но time×lat×lon ~ 12.6 млн float32
# это ~50 МБ, обычно ок. Чтобы ускорить выборки по точкам, можно загрузить в numpy:
da_np = da.values  # shape: (time, lat, lon), dtype float32

for r in st_pdf.itertuples(index=False):
    st = int(r.STATION)
    lat = float(r.LATITUDE)
    lon = float(r.LONGITUDE)

    # твой lon уже 0..180, но на всякий случай проверим
    if lon < float(lon_grid.min()) or lon > float(lon_grid.max()):
        continue

    ilat = nearest_index(lat_grid, lat)
    ilon = nearest_index(lon_grid, lon)

    series = da_np[:, ilat, ilon]  # shape (time,)

    # добавляем в список как 3 колонки
    rows.append(
        pd.DataFrame({
            "STATION": st,
            "DATE": times,
            "cmip_wind": series.astype(np.float32),
        })
    )

cmip_df = pd.concat(rows, ignore_index=True)

# === 5) сохраняем ===
cmip_pl = pl.from_pandas(cmip_df).with_columns(
    pl.col("DATE").cast(pl.Date)
)

cmip_pl.write_parquet(OUT_PATH, compression="zstd", statistics=True)

print("saved:", OUT_PATH, "rows:", cmip_pl.height, "stations:", cmip_pl.select(pl.col("STATION").n_unique()).item())
print(cmip_pl.head())


stations: 2419
shape: (5, 3)
┌────────────┬───────────┬───────────┐
│ STATION    ┆ LATITUDE  ┆ LONGITUDE │
│ ---        ┆ ---       ┆ ---       │
│ i64        ┆ f64       ┆ f64       │
╞════════════╪═══════════╪═══════════╡
│ 1006099999 ┆ 78.25     ┆ 22.816667 │
│ 1009099999 ┆ 80.65     ┆ 25.0      │
│ 1011099999 ┆ 80.066667 ┆ 31.5      │
│ 1016099999 ┆ 78.933333 ┆ 28.9      │
│ 1028099999 ┆ 74.516667 ┆ 19.016667 │
└────────────┴───────────┴───────────┘
cmip time points: 7617 from 2000-01-01 to 2020-11-13
saved: ../data/weatherstation_data/cmip_station_2000_2020_Russia.parquet rows: 18425523 stations: 2419
shape: (5, 3)
┌────────────┬────────────┬───────────┐
│ STATION    ┆ DATE       ┆ cmip_wind │
│ ---        ┆ ---        ┆ ---       │
│ i64        ┆ date       ┆ f32       │
╞════════════╪════════════╪═══════════╡
│ 1006099999 ┆ 2000-01-01 ┆ 9.432242  │
│ 1006099999 ┆ 2000-01-02 ┆ 13.564023 │
│ 1006099999 ┆ 2000-01-03 ┆ 13.871362 │
│ 1006099999 ┆ 2000-01-04 ┆ 17.04221  │
│ 1006099999

In [3]:
cmip_pl

STATION,DATE,cmip_wind
i64,date,f32
1006099999,2000-01-01,9.432242
1006099999,2000-01-02,13.564023
1006099999,2000-01-03,13.871362
1006099999,2000-01-04,17.04221
1006099999,2000-01-05,16.549023
…,…,…
99911099999,2016-12-27,2.94233
99911099999,2016-12-28,4.005349
99911099999,2016-12-29,5.510204


Unite train val and test and SAVE

In [ ]:
# cmip_eval = pl.read_parquet("../data/weatherstation_data/cmip_station_2017_2020_Russia.parquet")
# cmip_train  = pl.read_parquet("../data/weatherstation_data/cmip_station_2000_2016_Russia.parquet")

# cmip_all = pl.concat([cmip_train, cmip_eval])

# cmip_all.write_parquet(
#     "../data/weatherstation_data/cmip_station_2000_2020_Russia.parquet",
#     compression="zstd",
#     statistics=True
# )

Join df_obs ↔ cmip_station_daily

In [ ]:
import polars as pl
from datetime import date
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

DF_OBS_PATH   = "../data/weatherstation_data/df_obs_2000_2020_Russia.parquet"
CMIP_ST_PATH  = "../data/weatherstation_data/cmip_station_2000_2020_Russia.parquet"

df_obs  = pl.read_parquet(DF_OBS_PATH)
cmip_st = pl.read_parquet(CMIP_ST_PATH)

# inner join по (STATION, DATE)
df_eval = df_obs.join(cmip_st, on=["STATION", "DATE"], how="inner")

print("df_obs rows:", df_obs.height)
print("cmip_st rows:", cmip_st.height)
print("df_eval rows:", df_eval.height)
print("coverage:", df_eval.height / df_obs.height)


df_obs rows: 2454571
cmip_st rows: 2917192
df_eval rows: 2452735
coverage: 0.9992520077846597
